<p style="text-align:right" >
    <a href="https://www.iesb.br/home/" target="_blank">
    <img src="https://www.iesb.br/content/themes/iesb-chleba-themosis/assets/imagens/logo.png" width="100"/>
    </a>
</p>

# 🧠 Inteligência Artificial - IESB

**Tema da aula:** Laboratório 4 - Ciência de Dados e Pré-processamento
**Docente:** Prof. Rodrigo Gonçalves Pinto
**Ambiente:** Google Colab
**Conjunto de dados:** Passageiros do Titanic (891 registros, 12 colunas)

---

## 🎯 Objetivos de Aprendizagem

Neste laboratório, você vai:

1. **Carregar:** trazer um arquivo CSV da internet para dentro de um `DataFrame` do pandas.
2. **Inspecionar:** usar `head`, `shape`, `info` e `describe` para conhecer os dados antes de tocar neles.
3. **Diagnosticar:** localizar as células vazias e medir o tamanho do problema em cada coluna.
4. **Tratar:** decidir entre descartar linhas, descartar a coluna ou imputar um valor - e justificar a escolha.
5. **Codificar:** converter variáveis categóricas em números sem inventar uma ordem que não existe.
6. **Separar:** dividir treino e teste com semente fixa e proporção preservada.
7. **Padronizar:** colocar idade e tarifa na mesma unidade e demonstrar que isso muda o vizinho mais próximo.

---

## ⚠️ Antes de começar

> **Crie sua cópia:** vá em `Arquivo → Salvar uma cópia no Drive`. Trabalhe sempre na sua cópia, pois as edições no arquivo original não são salvas!

---

## 💡 Como aproveitar este notebook

- O conteúdo intercala **explicações teóricas** com **código comentado linha a linha**.
- Execute as células **na ordem** (`Shift + Enter`).
- Os **TODOs** têm validador automático: a célula avisa sozinha se a sua resposta está certa.
- Cada TODO se resolve com **uma única linha de código**. Se a sua solução ficou longa, provavelmente há um caminho mais direto.
- **Não pule as células de texto.** Elas contêm a justificativa que será cobrada na entrega.

## 🛠️ Pré-requisitos

- Conhecimentos básicos de Python: listas, dicionários, funções e `import`.
- Ter assistido à Aula 5 (método científico aplicado a dados, pré-processamento e separação treino/teste).

> *"Divida primeiro, ajuste no treino, aplique no teste."*

---
# Etapa 1 — Carregando os dados

## De onde vêm os dados

O conjunto do Titanic é um arquivo `.csv` (*comma-separated values*, ou valores separados por vírgula) hospedado na internet. É um arquivo de texto puro: a primeira linha traz os nomes das colunas, e cada linha seguinte traz um passageiro.

O pandas consegue ler esse arquivo direto do endereço, sem que você precise baixar nada. Quem faz isso é a função `pd.read_csv()`.

## As três importações do laboratório

| Biblioteca | Apelido | Para que serve aqui |
|---|---|---|
| `pandas` | `pd` | organizar os dados em tabela |
| `numpy` | `np` | fazer contas sobre vetores de números |
| `matplotlib.pyplot` | `plt` | desenhar os gráficos |

In [ ]:
# ---------------------------------------------------------------------
# IMPORTACOES — sempre na primeira celula do notebook.
#
# O "as pd" cria um APELIDO. Em vez de escrever pandas.read_csv(),
# escrevemos pd.read_csv(). E apenas uma economia de digitacao.
# ---------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Endereco do arquivo. Esta quebrado em duas linhas so para caber na tela;
# os parenteses fazem o Python juntar os dois pedacos em um texto so.
URL = ("https://raw.githubusercontent.com/datasciencedojo/datasets/"
       "master/titanic.csv")

# read_csv baixa o arquivo e devolve um DataFrame (uma tabela).
# "df" e o nome tradicional para essa variavel em qualquer projeto de dados.
df = pd.read_csv(URL)

print("Arquivo carregado com sucesso.")
print("Tipo do objeto:", type(df))

## As quatro chamadas de inspeção

Antes de modificar qualquer coisa, olhe. Estas quatro chamadas são obrigatórias no começo de todo notebook da disciplina:

| Chamada | O que responde |
|---|---|
| `df.head(n)` | como são as primeiras `n` linhas? |
| `df.shape` | quantas linhas e quantas colunas? |
| `df.info()` | qual o tipo de cada coluna e quantas células estão preenchidas? |
| `df.describe()` | qual o mínimo, o máximo, a média e os quartis das colunas numéricas? |

A mais importante das quatro é `df.info()`: é ela que revela **onde estão os buracos**.

In [ ]:
# head(5) mostra as 5 primeiras linhas. Serve para conferir se o arquivo
# foi lido corretamente e para ver o formato de cada coluna.
df.head(5)

In [ ]:
# shape devolve uma TUPLA no formato (numero_de_linhas, numero_de_colunas).
print("Formato da tabela:", df.shape)
print("Observacoes (passageiros):", df.shape[0])
print("Colunas:", df.shape[1])

print()

# columns devolve a lista de nomes das colunas.
print("Nomes das colunas:")
print(list(df.columns))

In [ ]:
# info() imprime um resumo: nome, quantidade de valores NAO nulos e tipo.
#
# COMO LER: se uma coluna mostra "891 non-null", ela esta completa.
# Se mostra menos que 891, faltam celulas. Procure as tres colunas
# incompletas na saida abaixo.
df.info()

## Interpretando a saída de `info()`

Repare que três colunas mostram menos de 891 valores preenchidos:

- **Age** — 714 preenchidos, portanto faltam 177
- **Cabin** — 204 preenchidos, portanto faltam 687
- **Embarked** — 889 preenchidos, portanto faltam 2

Repare também na coluna de tipos: `object` significa texto, `int64` significa número inteiro e `float64` significa número com casas decimais.

`Age` é `float64` e não `int64` porque existem bebês com idade fracionada (0,42 ano, por exemplo) - e também porque o pandas usa `float` sempre que há valores ausentes.

In [ ]:
# describe() resume apenas as colunas numericas.
#
# COMO LER cada linha da saida:
#   count -> quantos valores preenchidos
#   mean  -> media
#   std   -> desvio-padrao
#   min   -> menor valor
#   25%   -> primeiro quartil (25% dos dados estao abaixo dele)
#   50%   -> mediana
#   75%   -> terceiro quartil
#   max   -> maior valor
df.describe()

### Observe as duas linhas que interessam hoje

Compare `Age` e `Fare` na tabela acima:

| Coluna | mínimo | máximo | amplitude |
|---|---|---|---|
| `Age` | 0,42 | 80,00 | cerca de 80 |
| `Fare` | 0,00 | 512,33 | cerca de 512 |

A tarifa varre uma faixa **mais de seis vezes maior** que a idade. Guarde isso: é o problema que vamos resolver na Etapa 6.

Compare também `mean` e `50%` (a mediana) da coluna `Age`: a média é 29,70 e a mediana é 28,00. Quando a média é maior que a mediana, a distribuição tem uma **cauda à direita** - alguns valores altos puxam a média para cima.

---
# Etapa 2 — Diagnosticando os dados faltantes

## Vazio não é zero

Uma célula vazia (`NaN`, de *not a number*) significa **ausência de medida**. Zero significa **medida igual a zero**.

A diferença é concreta neste conjunto: existem passageiros com `Fare` igual a 0 - funcionários da companhia que viajaram sem pagar. Esse zero é informação legítima. Já `Age` vazio significa apenas que ninguém anotou a idade.

## Como o pandas detecta o vazio

- `df.isna()` devolve uma tabela do mesmo tamanho, preenchida com `True` onde a célula está vazia e `False` onde há valor.
- `.sum()` soma essa tabela por coluna. Como `True` vale 1 e `False` vale 0, a soma é exatamente a contagem de vazios.

In [ ]:
# isna() gera a tabela de True/False; sum() conta os True por coluna.
faltantes = df.isna().sum()

print("Celulas vazias por coluna:")
print(faltantes)

## 🛠️ TODO 1 — Medindo o buraco em percentual

A contagem absoluta não basta para decidir. Dizer que faltam 177 valores não informa muito; dizer que faltam **19,9%** informa.

Complete a função `percentual_faltantes` abaixo. Ela recebe uma coluna (uma `Series` do pandas) e deve devolver o percentual de células vazias, **arredondado para uma casa decimal**.

**Dica - as três peças que você precisa:**

- `serie.isna().sum()` conta os vazios da coluna
- `len(serie)` conta o total de linhas da coluna
- `round(numero, 1)` arredonda para uma casa decimal

**Resolve-se com uma única linha.**

In [ ]:
# =====================================================================
# TODO 1 — percentual de celulas vazias em uma coluna
#
# Receba uma Series e devolva o percentual de vazios com 1 casa decimal.
# Exemplo: se a coluna tem 891 linhas e 177 vazios, devolva 19.9
# =====================================================================

def percentual_faltantes(serie):
    # ESCREVA SUA RESPOSTA NA LINHA ABAIXO, no lugar de "return None"
    return None


# ------------------- teste automatico, nao precisa alterar -------------------
_r1 = percentual_faltantes(df["Age"])
_r2 = percentual_faltantes(df["Cabin"])

if _r1 is None or _r2 is None:
    print("TODO 1 ainda nao foi resolvido.")
elif _r1 == 19.9 and _r2 == 77.1:
    print("TODO 1 correto!")
    print("  Age   ->", _r1, "%")
    print("  Cabin ->", _r2, "%")
else:
    print("Resultado inesperado:")
    print("  Age   -> obtido", _r1, "| esperado 19.9")
    print("  Cabin -> obtido", _r2, "| esperado 77.1")
# -----------------------------------------------------------------------------

## O diagnóstico completo

Com a função pronta, dá para montar um relatório de uma vez só. A célula abaixo só roda corretamente se o TODO 1 estiver resolvido; caso contrário ela avisa e segue em frente, sem quebrar o notebook.

In [ ]:
# Percorre todas as colunas e monta um relatorio ordenado.
if percentual_faltantes(df["Age"]) is None:
    print("Resolva o TODO 1 antes de executar esta celula.")
else:
    relatorio = pd.DataFrame({
        "faltantes": df.isna().sum(),
        "percentual": [percentual_faltantes(df[c]) for c in df.columns],
    })
    # Mantem apenas as colunas que TEM problema, e ordena da pior para a melhor.
    relatorio = relatorio[relatorio["faltantes"] > 0]
    relatorio = relatorio.sort_values("percentual", ascending=False)
    print(relatorio)

## As três decisões

Cada uma das três colunas problemáticas recebe um tratamento diferente. **A decisão depende da proporção de vazios e da relevância da coluna** - não existe regra única.

| Coluna | Vazios | Decisão | Justificativa |
|---|---|---|---|
| `Age` | 177 (19,9%) | imputar com a mediana | atributo relevante e maioria preenchida; descartar linhas custaria um quinto do conjunto |
| `Cabin` | 687 (77,1%) | descartar a coluna | informação insuficiente; preencher inventaria 687 valores |
| `Embarked` | 2 (0,2%) | imputar com a moda | apenas duas linhas; qualquer escolha razoável resolve |

**Atenção:** ainda **não** vamos imputar nada nesta etapa. O valor a imputar só pode ser calculado depois da separação treino/teste, na Etapa 5. Aqui vamos apenas descartar a coluna `Cabin`, que é uma decisão que não depende de nenhum valor calculado.

In [ ]:
# drop() remove. columns=[...] diz QUAIS colunas remover.
# O resultado e uma nova tabela; por isso reatribuimos a df.
df = df.drop(columns=["Cabin", "Ticket", "PassengerId"])

print("Colunas restantes:", list(df.columns))
print("Novo formato:", df.shape)

### Por que `Ticket` e `PassengerId` saíram junto

- **`Ticket`** é texto livre sem formato único (`A/5 21171`, `STON/O2. 3101282`). Daria para extrair informação com esforço, mas não neste laboratório.
- **`PassengerId`** é um **identificador**: ele descreve a posição do passageiro na lista, não o passageiro. Se a lista estiver ordenada por algum critério, o modelo pode acabar aprendendo a posição em vez do fenômeno.

Note que `Name` **permanece**. Parece descartável, mas guarda o título social (`Mr.`, `Mrs.`, `Miss.`, `Master.`), que é informação de verdade escondida dentro de texto livre. Vamos usá-la no Desafio Final.

---
# Etapa 3 — Codificando variáveis categóricas

## O modelo não come texto

A coluna `Sex` guarda as palavras `male` e `female`. Nenhum modelo consegue multiplicar uma palavra por um coeficiente. Precisamos converter categoria em número.

**A pergunta é qual número**, e a resposta muda conforme a quantidade de categorias.

## Caso 1 — duas categorias: codificação binária

Com apenas dois valores possíveis, uma coluna de 0 e 1 basta. Dizer que `male` é 0 e `female` é 1 não cria nenhuma ordem falsa, porque com duas categorias só existe uma distância possível entre elas.

A ferramenta é o método `.map()`, que recebe um **dicionário** de tradução e troca cada valor pelo seu correspondente.

## 🛠️ TODO 2 — Codificação binária

Complete a função `codificar_binaria` abaixo. Ela recebe uma coluna e um dicionário de tradução, e deve devolver a coluna já traduzida.

**Dica:** o método `.map(dicionario)` faz exatamente isso, em uma chamada.

**Resolve-se com uma única linha.**

In [ ]:
# =====================================================================
# TODO 2 — traduzir os valores de uma coluna usando um dicionario
#
# Exemplo: codificar_binaria(coluna_sex, {"male": 0, "female": 1})
#          deve devolver a mesma coluna com 0 no lugar de "male"
#          e 1 no lugar de "female".
# =====================================================================

def codificar_binaria(serie, mapa):
    # ESCREVA SUA RESPOSTA NA LINHA ABAIXO
    return None


# ------------------- teste automatico, nao precisa alterar -------------------
_t1 = codificar_binaria(df["Sex"], {"male": 0, "female": 1})

# Segundo caso, com um dicionario DIFERENTE, para garantir que a funcao
# realmente usa o parametro "mapa" em vez de devolver um valor decorado.
_t2 = codificar_binaria(pd.Series(["S", "C", "S"]), {"S": 10, "C": 20})

if _t1 is None or _t2 is None:
    print("TODO 2 ainda nao foi resolvido.")
elif list(_t1.head(5)) == [0, 1, 1, 1, 0] and list(_t2) == [10, 20, 10]:
    print("TODO 2 correto!")
    print("  primeiros 5 valores de Sex ->", list(_t1.head(5)))
    print("  teste com outro dicionario ->", list(_t2))
else:
    print("Resultado inesperado:")
    print("  Sex     -> obtido", list(_t1.head(5)) if _t1 is not None else None,
          "| esperado [0, 1, 1, 1, 0]")
    print("  segundo -> obtido", list(_t2) if _t2 is not None else None,
          "| esperado [10, 20, 10]")
# -----------------------------------------------------------------------------

In [ ]:
# Aplica a codificacao de verdade na tabela, se o TODO 2 estiver pronto.
if codificar_binaria(df["Sex"], {"male": 0, "female": 1}) is None:
    print("Resolva o TODO 2 antes de executar esta celula.")
else:
    df["Sex"] = codificar_binaria(df["Sex"], {"male": 0, "female": 1})
    print("Coluna Sex codificada.")
    print(df[["Name", "Sex", "Age", "Survived"]].head(5))

## Caso 2 — três ou mais categorias: a armadilha da ordem falsa

A coluna `Embarked` tem três valores: `S` (Southampton), `C` (Cherbourg) e `Q` (Queenstown).

A tentação é traduzir para 0, 1 e 2. **Não faça isso.** Os números 0, 1 e 2 carregam duas afirmações que não são verdadeiras sobre portos:

1. **Ordem** - o modelo conclui que 2 é "maior" que 1, que é "maior" que 0.
2. **Distância** - o modelo conclui que a distância de `S` até `Q` é o dobro da distância de `S` até `C`.

Nenhuma das duas veio dos dados. Vieram da ordem em que escrevemos o dicionário.

## A solução: codificação um-de-n (*one-hot*)

Damos a cada categoria a sua própria coluna binária. Uma coluna vira três, e todas as categorias ficam à mesma distância entre si.

| Passageiro | `Embarked` antes | `Emb_S` | `Emb_C` | `Emb_Q` |
|---|---|---|---|---|
| Braund | S | 1 | 0 | 0 |
| Cumings | C | 0 | 1 | 0 |
| Moran | Q | 0 | 0 | 1 |

A função `pd.get_dummies()` faz isso em uma chamada.

In [ ]:
# Antes de codificar, precisamos preencher os 2 vazios de Embarked.
# Para uma variavel CATEGORICA nao existe mediana, entao usamos a MODA,
# que e o valor mais frequente.
#
# mode() devolve uma Series (pode haver empate); [0] pega o primeiro.
moda_embarque = df["Embarked"].mode()[0]
print("Valor mais frequente em Embarked:", moda_embarque)
print("Contagem por porto:")
print(df["Embarked"].value_counts())

# fillna() preenche as celulas vazias com o valor indicado.
df["Embarked"] = df["Embarked"].fillna(moda_embarque)
print()
print("Vazios restantes em Embarked:", df["Embarked"].isna().sum())

In [ ]:
# get_dummies transforma a coluna categorica em varias colunas binarias.
#   columns -> quais colunas transformar
#   prefix  -> como comecar o nome das novas colunas
#   dtype   -> pedimos int para termos 0 e 1 em vez de False e True
df = pd.get_dummies(df, columns=["Embarked"], prefix="Emb", dtype=int)

print("Novo formato:", df.shape)
print("Colunas agora:", list(df.columns))
print()
print(df[["Emb_S", "Emb_C", "Emb_Q"]].head(5))

### Confira a conta

A coluna `Embarked` saiu e três colunas nasceram. Se antes havia 9 colunas, agora há **9 - 1 + 3 = 11**. Confira com o `shape` impresso acima.

O custo do *one-hot* é o crescimento no número de colunas. Com três categorias, tudo bem. Com uma coluna de 50 categorias - estados de origem, por exemplo - o *one-hot* geraria 50 colunas, e aí existem alternativas melhores, que veremos na aula de agrupamento.

---
# Etapa 4 — Separando treino e teste

## A regra de ouro

> **Separe o teste antes de olhar para os dados. Tudo o que for decidido depois disso deve ser decidido usando apenas o treino.**

Repare que ainda **não** preenchemos a coluna `Age`. Isso foi de propósito: o valor a imputar (a mediana) precisa ser calculado **só no treino**, e para isso o treino precisa existir primeiro.

## O que `train_test_split` faz

A função embaralha as linhas e corta em dois pedaços. Ela recebe três parâmetros que importam:

| Parâmetro | O que faz | Por que importa |
|---|---|---|
| `test_size=0.2` | reserva 20% para teste | convenção; com 891 linhas, entre 20% e 30% é razoável |
| `random_state=42` | fixa a semente do sorteio | sem ele, cada execução dá um resultado diferente e o trabalho deixa de ser verificável |
| `stratify=y` | preserva a proporção do alvo nos dois lados | sem ele, a proporção de sobreviventes no teste varia vários pontos percentuais só por sorte |

**Cuidado com a ordem de retorno:** `X_treino, X_teste, y_treino, y_teste`. Trocar essa ordem é o erro de digitação mais comum do laboratório.

In [ ]:
from sklearn.model_selection import train_test_split

# X = tudo MENOS a coluna-alvo. y = so a coluna-alvo.
# Tiramos tambem Name, que e texto livre e nao entra em conta numerica.
X = df.drop(columns=["Survived", "Name"])
y = df["Survived"]

print("Formato de X:", X.shape)
print("Formato de y:", y.shape)
print("Atributos:", list(X.columns))

In [ ]:
# A separacao propriamente dita.
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.2,        # 20% para teste
    random_state=42,      # fixa o sorteio: sempre a mesma divisao
    stratify=y,           # preserva a proporcao de sobreviventes
)

print("Treino:", X_treino.shape[0], "passageiros")
print("Teste :", X_teste.shape[0], "passageiros")
print("Soma  :", X_treino.shape[0] + X_teste.shape[0])
print()

# Conferindo se a estratificacao funcionou:
# a media de uma coluna de 0 e 1 e a proporcao de 1.
print("Sobreviventes no treino: %.1f%%" % (y_treino.mean() * 100))
print("Sobreviventes no teste : %.1f%%" % (y_teste.mean() * 100))
print("Sobreviventes no total : %.1f%%" % (y.mean() * 100))

### O que você deve ter visto

- 712 passageiros no treino e 179 no teste, somando exatamente 891.
- As três proporções praticamente iguais: 38,3%, 38,5% e 38,4%. Isso **não** aconteceu por sorte - foi o `stratify`.

## Agora sim: imputando a idade, sem vazamento

O valor a imputar sai **apenas do treino**. Depois, o **mesmo** valor é aplicado aos dois lados.

Repare no detalhe: a mediana do treino é 28,5, mas a mediana dos 891 é 28,0. Se tivéssemos preenchido antes de separar, teríamos usado 28,0 - um número que só existe porque olhamos para o teste.

In [ ]:
# PASSO 1 — calcular o parametro SO no treino.
mediana_idade = X_treino["Age"].median()

print("Mediana da idade no TREINO ......:", mediana_idade)
print("Mediana da idade no conjunto todo:", df["Age"].median(), " <- NAO usar")
print("Mediana da idade no TESTE .......:", X_teste["Age"].median(), " <- NAO usar")

# PASSO 2 — aplicar o MESMO valor nos dois lados.
# O .copy() evita um aviso do pandas sobre alterar uma fatia da tabela.
X_treino = X_treino.copy()
X_teste = X_teste.copy()

X_treino["Age"] = X_treino["Age"].fillna(mediana_idade)
X_teste["Age"] = X_teste["Age"].fillna(mediana_idade)

print()
print("Vazios restantes no treino:", X_treino.isna().sum().sum())
print("Vazios restantes no teste :", X_teste.isna().sum().sum())

---
# Etapa 5 — Padronizando as escalas

## O defeito silencioso

Faltante gera erro. Texto gera erro. **Escala errada não gera erro nenhum:** o código roda, o resultado sai, e está pior do que poderia. Por isso este é o defeito que passa despercebido.

Compare novamente as duas colunas:

- `Age` vai de 0,42 a 80 - amplitude de cerca de 80
- `Fare` vai de 0 a 512,33 - amplitude de cerca de 512

Qualquer cálculo que compare as duas diretamente será dominado pela tarifa, simplesmente porque os números dela são maiores.

## A fórmula do escore z

$$z = \frac{valor - m\acute{e}dia}{desvio\text{-}padr\tilde{a}o}$$

- O **numerador** desloca a variável para que a média fique em zero.
- O **denominador** comprime ou estica a variável para que o desvio-padrão fique em um.

O resultado deixa de ser "anos" ou "libras" e passa a ser **"quantos desvios-padrão acima ou abaixo da média"**. É essa mudança de unidade que torna as colunas comparáveis.

## 🛠️ TODO 3 — A padronização

Complete a função `padronizar` abaixo. Ela recebe os valores, a média e o desvio-padrão, e deve devolver os valores padronizados.

**Dica:** é a fórmula do escore z, escrita em Python. Como `valores` é um vetor, a operação vale para todos os elementos de uma vez - é a vetorização que vimos na aula.

**Resolve-se com uma única linha.**

In [ ]:
# =====================================================================
# TODO 3 — padronizacao (escore z)
#
# Exemplo: padronizar(4.0, 29.81, 14.47) deve devolver aproximadamente
#          -1.7837, porque (4 - 29.81) / 14.47 = -1.7837...
# =====================================================================

def padronizar(valores, media, desvio):
    # ESCREVA SUA RESPOSTA NA LINHA ABAIXO
    return None


# ------------------- teste automatico, nao precisa alterar -------------------
_p1 = padronizar(4.0, 29.81, 14.47)

# Segundo caso com numeros totalmente diferentes, para garantir que a
# funcao calcula de verdade em vez de devolver um resultado decorado.
_p2 = padronizar(np.array([10.0, 20.0, 30.0]), 20.0, 10.0)

if _p1 is None or _p2 is None:
    print("TODO 3 ainda nao foi resolvido.")
elif abs(_p1 - (-1.7837)) < 0.001 and list(np.round(_p2, 4)) == [-1.0, 0.0, 1.0]:
    print("TODO 3 correto!")
    print("  padronizar(4.0, 29.81, 14.47) ->", round(_p1, 4))
    print("  padronizar([10, 20, 30], 20, 10) ->",
          [float(v) for v in np.round(_p2, 4)])
else:
    print("Resultado inesperado:")
    print("  caso 1 -> obtido", _p1, "| esperado aproximadamente -1.7837")
    print("  caso 2 -> obtido",
          [float(v) for v in np.round(_p2, 4)] if _p2 is not None else None,
          "| esperado [-1.0, 0.0, 1.0]")
# -----------------------------------------------------------------------------

## Aplicando, de novo sem vazamento

Exatamente a mesma disciplina da imputação: **média e desvio saem só do treino**, e os mesmos dois números são aplicados aos dois lados.

Se você recalcular a média no teste, o teste deixa de estar lacrado.

In [ ]:
if padronizar(4.0, 29.81, 14.47) is None:
    print("Resolva o TODO 3 antes de executar esta celula.")
else:
    colunas_numericas = ["Age", "Fare"]

    for coluna in colunas_numericas:
        # PASSO 1 — parametros calculados SO no treino.
        # ddof=0 usa o desvio-padrao populacional, que e o padrao do
        # StandardScaler do scikit-learn.
        media = X_treino[coluna].mean()
        desvio = X_treino[coluna].std(ddof=0)

        print("%-6s -> media do treino %8.2f | desvio do treino %8.2f"
              % (coluna, media, desvio))

        # PASSO 2 — os MESMOS numeros aplicados aos dois lados.
        X_treino[coluna] = padronizar(X_treino[coluna], media, desvio)
        X_teste[coluna] = padronizar(X_teste[coluna], media, desvio)

    print()
    print("Depois de padronizar, no treino:")
    print(X_treino[colunas_numericas].describe().loc[["mean", "std", "min", "max"]])

### Como conferir se deu certo

Na saída acima, no **treino**, a média de `Age` e `Fare` deve estar praticamente em 0 e o desvio praticamente em 1. É exatamente o que a fórmula promete.

No teste, a média **não** ficará exatamente em 0 - e isso está certo. O teste foi transformado com os números do treino, não com os dele próprio. Se a média do teste ficasse exatamente em 0, seria sinal de vazamento.

---
# Etapa 6 — Visualizando o efeito da padronização

Vamos reproduzir, no código, a demonstração feita em aula com três passageiros reais.

| Passageiro | Idade | Tarifa | Classe | Sobreviveu? |
|---|---|---|---|---|
| Dodge, Master. Washington | 4 | 81,86 | 1ª | sim |
| Maioni, Miss. Roberta | 16 | 86,50 | 1ª | sim |
| Goodwin, Master. Sidney Leonard | 1 | 46,90 | 3ª | não |

A pergunta: **de quem o pequeno Dodge está mais perto?**

Vamos medir com a distância euclidiana, que é o Teorema de Pitágoras aplicado a diferenças de atributos:

$$d(A, B) = \sqrt{(idade_A - idade_B)^2 + (tarifa_A - tarifa_B)^2}$$

In [ ]:
# Recarregamos o arquivo original so para pegar os valores CRUS
# desses tres passageiros (o df ja foi todo transformado).
bruto = pd.read_csv(URL)

nomes = ["Dodge, Master. Washington",
         "Maioni, Miss. Roberta",
         "Goodwin, Master. Sidney Leonard"]

tres = bruto[bruto["Name"].isin(nomes)][["Name", "Age", "Fare", "Pclass", "Survived"]]
print(tres.to_string(index=False))

In [ ]:
def distancia(p, q):
    """Distancia euclidiana entre dois pontos (idade, tarifa)."""
    # np.hypot(a, b) calcula a raiz de (a^2 + b^2) — e o Pitagoras pronto.
    return np.hypot(p[0] - q[0], p[1] - q[1])


# Valores CRUS: (idade, tarifa)
dodge_cru   = (4.0, 81.8583)
maioni_cru  = (16.0, 86.5)
goodwin_cru = (1.0, 46.9)

d_maioni = distancia(dodge_cru, maioni_cru)
d_goodwin = distancia(dodge_cru, goodwin_cru)

print("=== COM OS VALORES ORIGINAIS ===")
print("Dodge ate Maioni .:", round(d_maioni, 2))
print("Dodge ate Goodwin :", round(d_goodwin, 2))
print()
print("Vizinho mais proximo:",
      "Maioni (sobreviveu)" if d_maioni < d_goodwin else "Goodwin (nao sobreviveu)")

In [ ]:
if padronizar(4.0, 29.81, 14.47) is None:
    print("Resolva o TODO 3 antes de executar esta celula.")
else:
    # Parametros vindos do TREINO, os mesmos usados na Etapa 5.
    m_idade = bruto.loc[X_treino.index, "Age"].mean()
    s_idade = bruto.loc[X_treino.index, "Age"].std(ddof=0)
    m_tarifa = bruto.loc[X_treino.index, "Fare"].mean()
    s_tarifa = bruto.loc[X_treino.index, "Fare"].std(ddof=0)

    def para_z(p):
        return (padronizar(p[0], m_idade, s_idade),
                padronizar(p[1], m_tarifa, s_tarifa))

    dodge_z, maioni_z, goodwin_z = (para_z(dodge_cru),
                                    para_z(maioni_cru),
                                    para_z(goodwin_cru))

    print("Coordenadas padronizadas (idade, tarifa):")
    for nome, ponto in [("Dodge", dodge_z), ("Maioni", maioni_z),
                        ("Goodwin", goodwin_z)]:
        print("  %-8s (%6.2f ; %6.2f)" % (nome, ponto[0], ponto[1]))

    dz_maioni = distancia(dodge_z, maioni_z)
    dz_goodwin = distancia(dodge_z, goodwin_z)

    print()
    print("=== COM OS VALORES PADRONIZADOS ===")
    print("Dodge ate Maioni .:", round(dz_maioni, 3))
    print("Dodge ate Goodwin :", round(dz_goodwin, 3))
    print()
    print("Vizinho mais proximo:",
          "Maioni (sobreviveu)" if dz_maioni < dz_goodwin
          else "Goodwin (nao sobreviveu)")

### O que acabou de acontecer

Com os valores originais, o vizinho mais próximo de Dodge é **Maioni, que sobreviveu**.
Com os valores padronizados, o vizinho mais próximo passa a ser **Goodwin, que não sobreviveu**.

**A previsão se inverte.** Os dados são os mesmos; a única diferença foi a unidade de medida.

O motivo aparece na conta bruta: na distância até Goodwin, a diferença de tarifa ao quadrado vale 1222,08 e a de idade vale apenas 9,00. A tarifa respondeu por mais de 99% da distância, e a idade praticamente não participou.

In [ ]:
# Um grafico para fechar: os tres passageiros nas duas escalas.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

pontos_crus = [("Dodge", dodge_cru, "#C8102E"),
               ("Maioni", maioni_cru, "#1E4E79"),
               ("Goodwin", goodwin_cru, "#1B7F5A")]

for nome, (x, yv), cor in pontos_crus:
    ax1.scatter(x, yv, s=220, color=cor, zorder=5)
    ax1.annotate(nome, (x, yv), xytext=(8, 8), textcoords="offset points",
                 fontsize=11, fontweight="bold")
ax1.set_xlabel("idade (anos)")
ax1.set_ylabel("tarifa (libras)")
ax1.set_title("Escala original")
ax1.grid(color="#F4F6F8")
ax1.set_axisbelow(True)

if padronizar(4.0, 29.81, 14.47) is not None:
    for nome, ponto, cor in [("Dodge", dodge_z, "#C8102E"),
                             ("Maioni", maioni_z, "#1E4E79"),
                             ("Goodwin", goodwin_z, "#1B7F5A")]:
        ax2.scatter(ponto[0], ponto[1], s=220, color=cor, zorder=5)
        ax2.annotate(nome, ponto, xytext=(8, 8), textcoords="offset points",
                     fontsize=11, fontweight="bold")
    ax2.set_xlabel("idade padronizada (desvios-padrao)")
    ax2.set_ylabel("tarifa padronizada (desvios-padrao)")
    ax2.set_title("Depois de padronizar")
    ax2.grid(color="#F4F6F8")
    ax2.set_axisbelow(True)

plt.tight_layout()
plt.show()

---
# Etapa 7 — A linha de base

## Contra o que comparar?

Na próxima aula vamos treinar o primeiro modelo do semestre. Suponha que ele acerte 79% dos passageiros do teste. **Isso é bom?**

A pergunta não tem resposta sem uma referência. Precisamos saber quanto acertaria algo que **não aprendeu nada**. Esse valor chama-se **linha de base**, e calculá-lo é obrigatório antes de comemorar qualquer resultado.

In [ ]:
# Referencia 1 — dizer sempre "nao sobreviveu".
# Acerta exatamente a proporcao de mortes do conjunto.
base_maioria = (1 - y.mean()) * 100

# Referencia 2 — a regra que varias duplas escreveram em aula:
# "toda mulher sobreviveu, todo homem morreu".
# Lembre que Sex ja foi codificado: 1 = female, 0 = male.
palpite_sexo = df["Sex"]                      # 1 para mulher, 0 para homem
base_sexo = (palpite_sexo == y).mean() * 100

print("Chutar cara ou coroa .............: 50.0%")
print("Dizer que ninguem sobreviveu .....: %.1f%%" % base_maioria)
print("Regra 'toda mulher sobreviveu' ...: %.1f%%" % base_sexo)
print()
print("Nenhuma dessas tres aprendeu nada.")
print("Esta e a barra que o modelo da proxima aula tera de superar.")

---
# 🏁 Desafio Final (Entrega Obrigatória)

Esta seção vale a nota do laboratório. Ela tem duas partes: uma de código e uma de análise escrita.

---

## Parte 1 — Código: o pipeline sem vazamento

Você vai escrever uma função que executa **todo o pré-processamento na ordem correta**, do arquivo cru até as tabelas prontas.

A função recebe o DataFrame bruto e devolve `X_treino`, `X_teste`, `y_treino`, `y_teste`, já tratados.

**A ordem correta, em quatro passos:**

1. Separar treino e teste, **antes** de qualquer preparação.
2. Calcular os parâmetros de preparação usando **apenas o treino**.
3. Aplicar esses parâmetros ao treino.
4. Aplicar **exatamente os mesmos** parâmetros ao teste, sem recalcular nada.

O esqueleto abaixo já traz a estrutura. Você precisa completar **três linhas**, marcadas com `COMPLETE`.

In [ ]:
# =====================================================================
# TODO 4 — pipeline completo, sem vazamento de dados
#
# Complete as TRES linhas marcadas com COMPLETE.
# Use as funcoes que voce ja escreveu: padronizar() e codificar_binaria().
# =====================================================================

def preparar_sem_vazamento(bruto, semente=42):
    dados = bruto.copy()

    # --- limpeza que NAO depende de nenhum valor calculado ---
    dados = dados.drop(columns=["Cabin", "Ticket", "PassengerId", "Name"])
    dados["Sex"] = dados["Sex"].map({"male": 0, "female": 1})

    X = dados.drop(columns=["Survived"])
    y = dados["Survived"]

    # ----------------------------------------------------------------
    # PASSO 1 — separar PRIMEIRO.
    # COMPLETE: chame train_test_split com test_size=0.2,
    #           random_state=semente e stratify=y
    # ----------------------------------------------------------------
    X_tr, X_te, y_tr, y_te = None, None, None, None    # COMPLETE

    if X_tr is None:
        return None, None, None, None

    X_tr, X_te = X_tr.copy(), X_te.copy()

    # ----------------------------------------------------------------
    # PASSO 2 — parametros calculados SO no treino.
    # COMPLETE: a mediana da coluna Age, apenas no conjunto de treino
    # ----------------------------------------------------------------
    mediana = None                                     # COMPLETE

    moda_emb = X_tr["Embarked"].mode()[0]

    if mediana is None:
        return None, None, None, None

    # PASSO 3 e 4 — aplicar os MESMOS valores nos dois lados.
    for parte in (X_tr, X_te):
        parte["Age"] = parte["Age"].fillna(mediana)
        parte["Embarked"] = parte["Embarked"].fillna(moda_emb)

    # one-hot do porto, garantindo as mesmas colunas nos dois lados
    X_tr = pd.get_dummies(X_tr, columns=["Embarked"], prefix="Emb", dtype=int)
    X_te = pd.get_dummies(X_te, columns=["Embarked"], prefix="Emb", dtype=int)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    # ----------------------------------------------------------------
    # PASSO 5 — padronizar Age e Fare com media e desvio DO TREINO.
    # COMPLETE: dentro do laco, calcule media e desvio no TREINO
    #           e aplique padronizar() nos dois lados.
    # ----------------------------------------------------------------
    for coluna in ["Age", "Fare"]:
        media = None                                   # COMPLETE
        desvio = None                                  # COMPLETE
        if media is None or desvio is None:
            return None, None, None, None
        X_tr[coluna] = padronizar(X_tr[coluna], media, desvio)
        X_te[coluna] = padronizar(X_te[coluna], media, desvio)

    return X_tr, X_te, y_tr, y_te


# ------------------- teste automatico, nao precisa alterar -------------------
_a, _b, _c, _d = preparar_sem_vazamento(pd.read_csv(URL))

if _a is None:
    print("TODO 4 ainda nao foi resolvido.")
else:
    _ok_tam = (_a.shape[0] == 712 and _b.shape[0] == 179)
    _ok_vazio = (_a.isna().sum().sum() == 0 and _b.isna().sum().sum() == 0)
    _ok_media = abs(_a["Age"].mean()) < 1e-9 and abs(_a["Fare"].mean()) < 1e-9
    _ok_desvio = abs(_a["Age"].std(ddof=0) - 1) < 1e-9

    # Segunda semente: se estiver tudo certo, a divisao muda mas as
    # propriedades continuam valendo. Isso impede resposta decorada.
    _e, _f, _g, _h = preparar_sem_vazamento(pd.read_csv(URL), semente=7)
    _ok_outra = (_e is not None and abs(_e["Age"].mean()) < 1e-9
                 and _e.shape[0] == 712
                 and not _a.index.equals(_e.index))

    if _ok_tam and _ok_vazio and _ok_media and _ok_desvio and _ok_outra:
        print("TODO 4 correto!")
        print("  treino:", _a.shape, "| teste:", _b.shape)
        print("  vazios restantes:", _a.isna().sum().sum())
        print("  media de Age no treino: %.6f" % _a["Age"].mean())
        print("  desvio de Age no treino: %.6f" % _a["Age"].std(ddof=0))
        print("  media de Age no TESTE:  %.6f  (nao e zero, e esta certo)"
              % _b["Age"].mean())
    else:
        print("Resultado inesperado:")
        print("  tamanhos 712/179 ......:", _ok_tam)
        print("  sem celulas vazias ....:", _ok_vazio)
        print("  media zero no treino ..:", _ok_media)
        print("  desvio um no treino ...:", _ok_desvio)
        print("  funciona com outra semente:", _ok_outra)
# -----------------------------------------------------------------------------

---

## Parte 2 — Análise escrita

Responda às três perguntas abaixo **na célula de texto seguinte**, em português, com suas próprias palavras. Espera-se de 4 a 8 linhas por resposta.

**Pergunta 1.** Na Etapa 4, a mediana da idade no treino é 28,5 e no conjunto completo é 28,0. Explique por que usar 28,0 seria um erro, mesmo que a diferença de meio ano pareça irrelevante. Sua resposta deve deixar claro *que informação* estaria vazando e *em que direção* isso distorceria a avaliação final.

**Pergunta 2.** Na Etapa 6, o vizinho mais próximo de Dodge muda quando padronizamos. Escolha **outro** passageiro do conjunto, calcule o vizinho mais próximo dele antes e depois da padronização, e relate o que aconteceu. Se o vizinho **não** mudou, explique por que não - isso também é uma resposta válida e correta.

**Pergunta 3.** A regra "toda mulher sobreviveu" acerta 78,7% sem aprender nada. Um colega treina um modelo que acerta 79,5% no teste e conclui que o modelo funcionou. Você concorda? Justifique considerando o que a linha de base representa.

---

### ✍️ Escreva suas respostas aqui

*(dê um duplo clique nesta célula para editar)*

**Resposta 1.**

_(escreva aqui)_

**Resposta 2.**

_(escreva aqui — cole também o código que você usou para escolher e testar o passageiro)_

**Resposta 3.**

_(escreva aqui)_

In [ ]:
# Espaco livre para o codigo da Pergunta 2.
# Sugestao: reaproveite a funcao distancia() da Etapa 6 e percorra
# o conjunto procurando o vizinho mais proximo do passageiro escolhido.

---

## ✅ Checklist de Entrega

Antes de postar, confira cada item:

- [ ] TODO 1 imprime `TODO 1 correto!`
- [ ] TODO 2 imprime `TODO 2 correto!`
- [ ] TODO 3 imprime `TODO 3 correto!`
- [ ] TODO 4 imprime `TODO 4 correto!`
- [ ] As três perguntas da Parte 2 estão respondidas, com suas próprias palavras
- [ ] Executei `Ambiente de execução → Reiniciar sessão e executar tudo` e **nenhuma célula quebrou**
- [ ] O arquivo foi renomeado para `lab4_SEUNOME.ipynb`
- [ ] Postei no Google Sala de Aula até **domingo, 06/09, às 23h59**

> **Reprodutibilidade vale ponto.** Notebook que não roda do zero perde nota mesmo com todos os TODOs corretos.

---

## Resumo de Fixação (De/Para)

| Quando você quer... | Use... |
|---|---|
| ler um CSV da internet | `pd.read_csv(url)` |
| ver o tamanho da tabela | `df.shape` |
| descobrir onde faltam valores | `df.isna().sum()` |
| remover uma coluna | `df.drop(columns=["nome"])` |
| preencher células vazias | `df["col"].fillna(valor)` |
| descobrir o valor mais frequente | `df["col"].mode()[0]` |
| traduzir categorias em números | `df["col"].map({...})` |
| criar colunas binárias por categoria | `pd.get_dummies(df, columns=[...])` |
| separar treino e teste | `train_test_split(X, y, test_size=..., random_state=..., stratify=y)` |
| padronizar uma coluna | `(valores - média) / desvio` |
| contar sobreviventes por grupo | `df.groupby("col")["Survived"].mean()` |

---

## Para a Próxima Aula

**09 de setembro — k-vizinhos mais próximos e modelos de regressão.**

A tabela que você acabou de preparar é exatamente o que o primeiro modelo do semestre vai receber. E a conta de distância da Etapa 6 - aquela que fizemos com três passageiros - vira o modelo inteiro: o k-vizinhos classifica um passageiro perguntando o que aconteceu com os `k` passageiros mais parecidos com ele.

**Leitura recomendada:** GÉRON, A. *Mãos à obra: aprendizado de máquina com Scikit-Learn, Keras e TensorFlow*, capítulo 2.

**Pergunta para pensar:** se o k-vizinhos olha para os `k` vizinhos mais próximos, o que acontece quando `k = 1`? E quando `k` é igual ao número total de passageiros do treino?